> 在线讲义：[第 11 章 · 读原版源码：The Annotated Transformer 里藏着哪些和论文不一样的地方](https://wikieden.github.io/beyond-transformer/lessons/11-annotated-transformer.html) · 本书《超越 Transformer》

# 第 11 章 · 读原版源码：The Annotated Transformer 里藏着哪些和论文不一样的地方

配套讲义：[在线版](https://wikieden.github.io/beyond-transformer/lessons/11-annotated-transformer.html) · 仓库里 `beyond-transformer/notes/lessons/11-annotated-transformer.md`（附录最后一节"代码对照"）

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 Python 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 NumPy，不需要 GPU。练习题不在这里算，先在讲义里自己做。

In [ ]:
import numpy as np

np.set_printoptions(precision=4, suppress=True)  # 打印保留 4 位小数，不用科学计数法
print("numpy", np.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 Post-LN vs Pre-LN | 两种残差 | `post_ln`、`pre_ln` | [−1.34, …] vs [−3.02, …, 8.02] |
| 第三步 两种 LayerNorm | n − 1 vs n | `layer_norm_annotated`、`layer_norm_torch` | −1.1619 vs −1.3416 |
| 第四步 label smoothing | 目标分布 + KL | `smoothed_target()`、`kl_loss()`、`confidence_curve()` | 最低点在 0.900 |
| 第五步 Noam 调度 | min 两项 | `rate(step)` | 峰值 6.99e-4 |

它的 LayerNorm 用无偏标准差、eps 加在根号外；label smoothing 用 KL 散度和平滑后的目标分布；
学习率 rate() = d^{-0.5}·min(step^{-0.5}, step·warmup^{-1.5})。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

**函数 `post_ln`**：论文：LayerNorm(x + Sublayer(x))。

In [ ]:
def post_ln(x, sub):
    """论文：LayerNorm(x + Sublayer(x))。"""
    return layer_norm_torch(x + sub(x))

**函数 `pre_ln`**：Annotated 的 SublayerConnection：x + Sublayer(LayerNorm(x))。

In [ ]:
def pre_ln(x, sub):
    """Annotated 的 SublayerConnection：x + Sublayer(LayerNorm(x))。"""
    return x + sub(layer_norm_torch(x))

**函数 `layer_norm_annotated`**：源码 L315–L328：(x − mean) / (std + eps)，std 是无偏的（除以 n − 1）。

In [ ]:
def layer_norm_annotated(x, eps=1e-6):
    """源码 L315–L328：(x − mean) / (std + eps)，std 是无偏的（除以 n − 1）。"""
    return (x - x.mean()) / (x.std(ddof=1) + eps)

**函数 `layer_norm_torch`**：nn.LayerNorm：(x − mean) / √(var + eps)，var 是有偏的（除以 n）。

In [ ]:
def layer_norm_torch(x, eps=1e-5):
    """nn.LayerNorm：(x − mean) / √(var + eps)，var 是有偏的（除以 n）。"""
    return (x - x.mean()) / np.sqrt(x.var() + eps)

**函数 `smoothed_target`**：源码 L1160–L1166：先每格填 smoothing/(size − 2)，正确格填 1 − smoothing，padding 格清零。

In [ ]:
def smoothed_target(size=5, target=2, smoothing=0.1, padding_idx=0):
    """源码 L1160–L1166：先每格填 smoothing/(size − 2)，正确格填 1 − smoothing，padding 格清零。"""
    d = np.full(size, smoothing / (size - 2))
    d[target] = 1 - smoothing
    d[padding_idx] = 0
    return d

**函数 `kl_loss`**：nn.KLDivLoss(reduction='sum')：Σ q (log q − log p)，q = 0 的格不算。

In [ ]:
def kl_loss(pred, target_dist):
    """nn.KLDivLoss(reduction='sum')：Σ q (log q − log p)，q = 0 的格不算。"""
    m = target_dist > 0
    return float((target_dist[m] * (np.log(target_dist[m]) - np.log(pred[m]))).sum())

**函数 `confidence_curve`**：源码 L1232 的例子：预测 [0, x/d, 1/d, 1/d, 1/d]（d = x + 3），正确类 1。x 越大越“自信”。

In [ ]:
def confidence_curve(xs=range(1, 100), smoothing=0.1):
    """源码 L1232 的例子：预测 [0, x/d, 1/d, 1/d, 1/d]（d = x + 3），正确类 1。x 越大越“自信”。"""
    q = smoothed_target(5, 1, smoothing)
    out = []
    for x in xs:
        d = x + 3
        p = np.array([1e-12, x / d, 1 / d, 1 / d, 1 / d])
        out.append((x, x / d, kl_loss(p, q)))
    return out

**函数 `rate`**：源码 L1058：AIAYN 式 (3)。

In [ ]:
def rate(step, model_size=512, factor=1.0, warmup=4000):
    """源码 L1058：AIAYN 式 (3)。"""
    step = max(step, 1)
    return factor * model_size**-0.5 * min(step**-0.5, step * warmup**-1.5)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("第一步 SublayerConnection：源码是先 norm")
x = np.array([1.0, 2.0, 3.0, 4.0])
sub = lambda v: 3 * v  # noqa: E731  一个让数变大的“子层”
print(f"  x = {x.tolist()}，子层 = 乘 3")
print(f"  论文 Post-LN：{np.round(post_ln(x, sub), 4).tolist()}（输出被 norm 住）")
print(f"  源码 Pre-LN： {np.round(pre_ln(x, sub), 4).tolist()}（主干上的 x 原样保留）")
print("\n第二步 源码的 LayerNorm 和 nn.LayerNorm 不完全一样")
print(f"  [1, 2, 3, 4]：源码 {np.round(layer_norm_annotated(x), 4).tolist()}；nn.LayerNorm {np.round(layer_norm_torch(x), 4).tolist()}")

In [ ]:
print("\n第三步 label smoothing")
q = smoothed_target()
print(f"  词表 5、padding 0、正确类 2、smoothing 0.1：目标分布 {np.round(q, 4).tolist()}")
for p in ([0.01, 0.05, 0.9, 0.02, 0.02], [0.001, 0.001, 0.996, 0.001, 0.001]):
    print(f"  预测 {p}：KL loss {kl_loss(np.array(p), q):.4f}")
curve = confidence_curve()
best = min(curve, key=lambda r: r[2])
print(f"  正确类概率从 0.25 涨到 0.97：loss 在 x = {best[0]}（概率 {best[1]:.3f}）最小，再自信 loss 反而变大（x = 99：{curve[-1][2]:.4f}）")

In [ ]:
print("\n第四步 学习率 rate()（d = 512、warmup = 4000）")
for s in (1, 1000, 4000, 16000, 100000):
    print(f"  step {s:6d}：{rate(s):.2e}")

## 3. 代码里才看得到的

- **`x.std()` 的默认值**：PyTorch 的 `Tensor.std()` 默认无偏（除以 n − 1），NumPy 的 `np.std()` 默认有偏（除以 n）。同一个名字，两个库默认不一样，这里用 `ddof=1` 对齐源码。
- **KLDivLoss 吃的是 log 概率**：源码的 `Generator` 输出 log_softmax，`LabelSmoothing` 直接把它传给 `KLDivLoss`；q = 0 的格（padding）不贡献 loss。

## 4. 自己改一改

- 把 `confidence_curve` 的 smoothing 改成 0.3，最低点移到哪？
- 把 `pre_ln` 叠三层（每层子层都乘 3），主干上的数长到多大？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> 一个类一个零件，L230 读到 L1172，前作从头拼一遍。
> 残差先 norm 再子层，源码偷懒成了 Pre-LN，最后补一个 norm。
> 源码 LayerNorm 无偏 std、ε 在根号外，宽了才和官方一样。
> 平滑目标零点九，其余均分；自信过了九成反而罚。
> Noam 调度：线性升到 warmup，之后一除根号步。